# Notebook 05: Embeddings and Topic Modeling

**Pipeline layer G.** Supplementary semantic representation of the corpus. Reads `movies_union.parquet` and `stockiness_long_classified.parquet`. Produces voyage-4-large embeddings of every plot and every classified narrative element, plus BERTopic and GMM clusterings of the plot embeddings, plus a per-genre semantic-validation table that checks whether the framework's top stock elements per genre also sit close to their genre's semantic centroid.

## Purpose

Everything in the pipeline so far is built on **counts**. Notebook 02 extracts elements by counting tokens. Notebook 03 scores stockiness by counting films, instances, and genres. Notebook 03b classifies elements with an LLM that reads metadata. Notebook 03c builds co-occurrence networks from film-level counts. The framework is a sophisticated counting machine.

This notebook offers a **completely different representation** of the same corpus. Instead of counting, it embeds each plot and each element into a high-dimensional semantic vector using voyage-4-large, then asks whether the structure the framework identifies through counts also appears under purely semantic representation.

Two doctrinal-validity questions are answered.

**Question one: do films cluster into recognizable genre-like groups when you look at their semantic embeddings without knowing the genre labels?** If you embed all 45,958 films, throw away the genre tags, and let BERTopic and a Gaussian-mixture model discover clusters, do the discovered clusters align with the seventeen canonical genres? If they do, the genre labels we have been using throughout the pipeline are picking up real semantic structure in the corpus, not just label conventions. This validates the entire framework's use of genre as the unit of stockiness analysis.

**Question two: are the elements the framework identifies as central to a genre also semantically close to that genre's centroid?** For each canonical genre, compute the mean embedding of films tagged with that genre (the genre's semantic centroid). Compute the cosine similarity between each top-stockiness element's embedding and the genre's centroid. If the framework's top stock elements for Western (sheriff, ranch, saloon, posse) are semantically closer to the Western centroid than random elements are, the framework's count-based scoring is independently confirmed by a completely different semantic representation.

This is a **sanity check**, not a headline finding. The framework's stockiness scores from Notebooks 03 and 03b are the primary doctrinal claim. Notebook 05 is the layer that says: when you look at the corpus through a fundamentally different lens, do you see the same structure the framework sees? Strong agreement strengthens the framework's empirical foundation; partial disagreement is doctrinally interesting in its own right and is documented in the report.

## Method

**Plot embeddings.** Each of the 45,958 plot summaries is sent to the voyage-4-large embedding endpoint with `input_type='document'`. The model returns a fixed-dimensional vector per plot. Plots are batched up to 128 per request to amortize API overhead. The full embedding matrix is saved as a numpy file alongside a parquet of `(union_id, embedding_index)` so we can reconnect to the corpus metadata.

**Element embeddings.** Each of the 41,186 unique `(element, element_type)` pairs from `stockiness_long_classified.parquet` is embedded the same way. The element string is the input, with no surrounding context. Stop after filtering to `is_narrative_element=True` would be one choice; for completeness we embed every pair and let downstream analysis filter, since the embedding cost is trivial for the additional ~14K filtered pairs.

**BERTopic clustering on plot embeddings.** BERTopic discovers topic clusters by combining UMAP dimensionality reduction with HDBSCAN clustering and c-TF-IDF topic representation. The output is a topic assignment per plot plus a topic-summary table with the most distinctive words per discovered topic. We set `min_cluster_size=100` (about 0.2% of the corpus) to keep the topics interpretable rather than fragmenting into many tiny clusters.

**Gaussian-mixture-model clustering on plot embeddings.** As an alternative clustering approach, we fit a GMM with the number of components chosen by Bayesian information criterion (BIC). We sweep n_components from 8 to 30 and pick the value that minimizes BIC. The GMM provides soft cluster assignments (per-plot membership probabilities) that complement BERTopic's hard assignments.

**Genre-cluster alignment.** For both clusterings, compute the contingency table between discovered clusters and the seventeen canonical genres. Report adjusted Rand index, normalized mutual information, and a confusion-matrix view. Clean alignment means the embeddings recover the genre structure; weak alignment means the genres are noisier than the embeddings suggest.

**Per-genre semantic centroids.** For each canonical genre g, compute the mean plot embedding over films tagged with g (the centroid). For each element in the corpus, compute the cosine similarity between the element embedding and each genre's centroid. The headline check: for each canonical genre, take the top-50 narrative elements by ensemble stockiness score, compute the mean cosine similarity to the genre's centroid, and compare against a random-50-elements baseline. The top-stock elements should be substantially closer to the centroid than random.

## Inputs

- `v4_rebuild/01c_union/movies_union.parquet` — 45,958 films with cleaned plot text
- `v4_rebuild/03b_classification/stockiness_long_classified.parquet` — 41,186 classified `(element, element_type)` pairs with ensemble scores per genre

## Outputs

- `v4_rebuild/05_embeddings/plot_embeddings.npy` — (45,958, embedding_dim) numpy float32 matrix
- `v4_rebuild/05_embeddings/plot_embeddings_index.parquet` — (union_id, embedding_row_index) mapping
- `v4_rebuild/05_embeddings/element_embeddings.npy` — (41,186, embedding_dim) matrix
- `v4_rebuild/05_embeddings/element_embeddings_index.parquet` — (element, element_type, embedding_row_index)
- `v4_rebuild/05_embeddings/bertopic_topics.parquet` — per-plot topic assignment, topic name, topic probability
- `v4_rebuild/05_embeddings/gmm_clusters.parquet` — per-plot cluster assignment + membership probabilities for the BIC-selected number of clusters
- `v4_rebuild/05_embeddings/genre_centroids.parquet` — per-genre centroid stats and per-element cosine similarities
- `v4_rebuild/05_embeddings/cluster_genre_alignment.parquet` — adjusted Rand, NMI, and confusion-matrix summaries
- `v4_rebuild/05_embeddings/semantic_validation_report.txt` — headline findings and per-genre breakdowns

## Cost and runtime

- voyage-4-large embeddings: roughly $2-3 total for plots plus elements. The exact price depends on Voyage's per-token rate at run time; the corpus has about 11.5M plot tokens and 200K element tokens.
- BERTopic + GMM: pure CPU clustering on the embedding matrix. Roughly 15-30 minutes.
- Genre-alignment and centroid computation: a few minutes of pandas / numpy work.

Total wall time: 1-2 hours, dominated by the API embedding calls. No GPU needed.

In [14]:
# ----------------------------------------------------------------------
# Setup and configuration
# ----------------------------------------------------------------------
!pip install -q voyageai bertopic umap-learn hdbscan scikit-learn

import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- Google Drive -----------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# --- Paths ------------------------------------------------------------
ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
            'DGK Lab - Scene A Faire Computational')
V4 = ROOT / 'Analysis/v4_rebuild'

PATH_MOVIES   = V4 / '01c_union'          / 'movies_union.parquet'
PATH_ENRICHED = V4 / '03b_classification' / 'stockiness_long_classified.parquet'

OUT_DIR = V4 / '05_embeddings'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_PLOT_EMB       = OUT_DIR / 'plot_embeddings.npy'
PATH_PLOT_INDEX     = OUT_DIR / 'plot_embeddings_index.parquet'
PATH_ELEM_EMB       = OUT_DIR / 'element_embeddings.npy'
PATH_ELEM_INDEX     = OUT_DIR / 'element_embeddings_index.parquet'
PATH_BERTOPIC       = OUT_DIR / 'bertopic_topics.parquet'
PATH_GMM            = OUT_DIR / 'gmm_clusters.parquet'
PATH_CENTROIDS      = OUT_DIR / 'genre_centroids.parquet'
PATH_ALIGNMENT      = OUT_DIR / 'cluster_genre_alignment.parquet'
PATH_REPORT         = OUT_DIR / 'semantic_validation_report.txt'

# --- API key (Colab Secrets only — never hardcode) --------------------
from google.colab import userdata
try:
    VOYAGE_API_KEY = userdata.get('SAF_Voyage_API')
    assert VOYAGE_API_KEY, 'empty'
except Exception:
    raise RuntimeError(
        'SAF_Voyage_API not found in Colab Secrets. Sign up at '
        'voyageai.com, generate an API key, add it as a Colab Secret '
        'named SAF_Voyage_API, and enable notebook access. Do not paste '
        'the key into a cell.'
    )

import voyageai
voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)

# --- Run configuration ------------------------------------------------
MODEL_NAME       = 'voyage-4-large'
BATCH_SIZE_PLOTS = 32         # voyage-4-large allows larger batches; smaller is safer for long plots
BATCH_SIZE_ELEMS = 128        # short element strings, large batches fine
MAX_TOKENS_PER_INPUT = 8000   # voyage-4-large max; truncate longer plots
CHECKPOINT_EVERY = 1000        # flush partial embedding matrix every N items
REQUEST_SLEEP    = 0.1        # seconds between batches (rate-limit hygiene)
REQUEST_RETRIES  = 5

# Genre list, identical to the rest of the pipeline
CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western',
]

# GMM BIC sweep range and adaptive-stop patience.
# The first run of this notebook used range(8, 31), but the BIC kept
# decreasing through n=30 with no plateau, meaning we hit the upper bound
# without finding a true minimum. The patched version extends the upper
# bound and adds adaptive stopping: if the BIC has not improved for
# GMM_PATIENCE consecutive steps, we stop and report the running minimum.
# This way the sweep finds the actual BIC-optimal n if one exists within
# the range, and reports a clear "kept descending" warning if not.
GMM_N_MIN = 8
GMM_N_MAX = 80
GMM_PATIENCE = 8   # stop after this many consecutive non-improvements
GMM_N_RANGE = list(range(GMM_N_MIN, GMM_N_MAX + 1))

# BERTopic minimum cluster size. 100 ≈ 0.2% of the corpus; small enough to
# capture genre sub-conventions, large enough to keep topics interpretable.
BERTOPIC_MIN_CLUSTER_SIZE = 100

print('Setup complete.')
print(f'  Model     : {MODEL_NAME}')
print(f'  Output dir: {OUT_DIR}')
print(f'  BERTopic min cluster size: {BERTOPIC_MIN_CLUSTER_SIZE}')
print(f'  GMM components sweep     : {GMM_N_MIN} to {GMM_N_MAX}, patience={GMM_PATIENCE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.
  Model     : voyage-4-large
  Output dir: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/05_embeddings
  BERTopic min cluster size: 100
  GMM components sweep     : 8 to 80, patience=8


In [15]:
# ----------------------------------------------------------------------
# Load inputs
# ----------------------------------------------------------------------
print('Loading movies_union.parquet...')
movies = pd.read_parquet(PATH_MOVIES)
movies['genres_canonical'] = movies['genres_canonical'].apply(
    lambda gs: list(gs) if hasattr(gs, '__len__') and not isinstance(gs, str) else []
)
print(f'  {len(movies):,} films')

# Defensive filter: surprisal computation in 04 dropped films with fewer
# than 20 plot words. Apply the same filter here for consistency, even
# though the embedding API can handle very short text.
movies['plot_word_count'] = movies['plot'].fillna('').str.split().str.len()
movies = movies[movies['plot_word_count'] >= 20].reset_index(drop=True)
print(f'  {len(movies):,} films after the same-as-04 length filter')

print('\nLoading stockiness_long_classified.parquet...')
stk_cls = pd.read_parquet(PATH_ENRICHED)
print(f'  {len(stk_cls):,} (element, type, genre) cells')

# Unique (element, element_type) pairs are the elements we embed.
elements = (stk_cls[['element','element_type','is_narrative_element','element_taxonomy']]
            .drop_duplicates(subset=['element','element_type'])
            .sort_values(['element','element_type'])
            .reset_index(drop=True))
print(f'  {len(elements):,} unique (element, element_type) pairs to embed')

Loading movies_union.parquet...
  45,958 films
  45,958 films after the same-as-04 length filter

Loading stockiness_long_classified.parquet...
  456,159 (element, type, genre) cells
  40,877 unique (element, element_type) pairs to embed


In [16]:
# ----------------------------------------------------------------------
# Phase A — Embed plots via voyage-4-large
#
# Batches of BATCH_SIZE_PLOTS plots per request. Each batch:
#   1. Truncate each plot to MAX_TOKENS_PER_INPUT tokens (approximate
#      via word count × 1.3, a conservative ratio).
#   2. Call voyage_client.embed(texts, model=MODEL_NAME, input_type='document').
#   3. Append the returned vectors to the in-memory list.
#   4. Every CHECKPOINT_EVERY items, flush to disk.
#
# CHECKPOINT: PATH_PLOT_EMB + PATH_PLOT_INDEX. Resume support: if the
# index parquet exists, we read which union_ids are done and skip them.
# ----------------------------------------------------------------------

def voyage_embed(texts, retries=REQUEST_RETRIES):
    """Embed a batch with retries on rate-limit/transient errors."""
    last_err = None
    delay = 2.0
    for attempt in range(retries):
        try:
            result = voyage_client.embed(
                texts=texts,
                model=MODEL_NAME,
                input_type='document',
                truncation=True,
            )
            return result.embeddings
        except Exception as e:
            last_err = e
            time.sleep(delay)
            delay = min(delay * 2, 60)
    raise RuntimeError(f'voyage embed failed after {retries} attempts: {last_err}')


def truncate_words(text, max_words):
    """Conservative truncation by word count. voyage-4-large will also
    truncate at the token level via truncation=True, so this is a defensive
    pre-trim for the very longest plots."""
    words = text.split()
    if len(words) <= max_words:
        return text
    return ' '.join(words[:max_words])


# Resume from checkpoint if present
if PATH_PLOT_INDEX.exists() and PATH_PLOT_EMB.exists():
    print(f'Loading existing plot-embedding checkpoint: {PATH_PLOT_INDEX}')
    plot_index = pd.read_parquet(PATH_PLOT_INDEX)
    plot_emb_existing = np.load(PATH_PLOT_EMB)
    assert len(plot_index) == len(plot_emb_existing), 'index/matrix length mismatch'
    done_ids = set(plot_index['union_id'].astype(int))
    print(f'  resuming: {len(done_ids):,} films already embedded')
    embeddings_list = [plot_emb_existing]
    index_records = plot_index.to_dict('records')
else:
    done_ids = set()
    embeddings_list = []
    index_records = []
    print('No plot-embedding checkpoint found; starting from scratch.')

remaining = movies[~movies['union_id'].isin(done_ids)].reset_index(drop=True)
print(f'{len(remaining):,} plots to embed')

# Estimate cost (rough — voyage charges per token, ~$0.18/M for voyage-4-large)
est_tokens = remaining['plot_word_count'].sum() * 1.3
print(f'  estimated token volume: ~{est_tokens/1e6:.1f}M tokens')
print(f'  estimated cost        : ~${est_tokens/1e6 * 0.18:.2f} (at $0.18/M tokens)')

# Cap each plot at MAX_TOKENS_PER_INPUT / 1.3 words as a defensive pre-trim
max_words = int(MAX_TOKENS_PER_INPUT / 1.3)

n_total = len(remaining)
since_flush = 0
new_batch_embs = []
new_batch_index = []

pbar = tqdm(range(0, n_total, BATCH_SIZE_PLOTS), desc='plot embedding')
for batch_start in pbar:
    batch = remaining.iloc[batch_start : batch_start + BATCH_SIZE_PLOTS]
    texts = [truncate_words(t, max_words) for t in batch['plot'].tolist()]
    embs = voyage_embed(texts)
    new_batch_embs.extend(embs)
    for _, row in batch.iterrows():
        new_batch_index.append({
            'union_id': int(row['union_id']),
            'embedding_row_index': len(index_records) + len(new_batch_index),
        })
    since_flush += len(batch)
    time.sleep(REQUEST_SLEEP)

    if since_flush >= CHECKPOINT_EVERY:
        embeddings_list.append(np.array(new_batch_embs, dtype=np.float32))
        index_records.extend(new_batch_index)
        flushed = np.concatenate(embeddings_list, axis=0)
        np.save(PATH_PLOT_EMB, flushed)
        pd.DataFrame(index_records).to_parquet(PATH_PLOT_INDEX, index=False)
        embeddings_list = [flushed]
        new_batch_embs = []
        new_batch_index = []
        since_flush = 0
        pbar.set_postfix(flushed=len(flushed))

# Final flush
if new_batch_embs:
    embeddings_list.append(np.array(new_batch_embs, dtype=np.float32))
    index_records.extend(new_batch_index)
plot_embeddings = np.concatenate(embeddings_list, axis=0)
np.save(PATH_PLOT_EMB, plot_embeddings)
pd.DataFrame(index_records).to_parquet(PATH_PLOT_INDEX, index=False)

print(f'\nPhase A complete.')
print(f'  plot embeddings: {plot_embeddings.shape}  (films × embedding_dim)')
print(f'  saved: {PATH_PLOT_EMB} and {PATH_PLOT_INDEX}')

Loading existing plot-embedding checkpoint: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/05_embeddings/plot_embeddings_index.parquet
  resuming: 45,958 films already embedded
0 plots to embed
  estimated token volume: ~0.0M tokens
  estimated cost        : ~$0.00 (at $0.18/M tokens)


plot embedding: 0it [00:00, ?it/s]


Phase A complete.
  plot embeddings: (45958, 1024)  (films × embedding_dim)
  saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/05_embeddings/plot_embeddings.npy and /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/05_embeddings/plot_embeddings_index.parquet


In [17]:
# ----------------------------------------------------------------------
# Phase B — Embed elements via voyage-4-large
#
# Each (element, element_type) pair is embedded as the bare element
# string (the element_type is metadata, not part of the embedded text).
# The downstream similarity analysis joins back on the (element, type)
# tuple via the index parquet.
#
# CHECKPOINT: PATH_ELEM_EMB + PATH_ELEM_INDEX. Resume support identical
# to Phase A.
# ----------------------------------------------------------------------

if PATH_ELEM_INDEX.exists() and PATH_ELEM_EMB.exists():
    print(f'Loading existing element-embedding checkpoint: {PATH_ELEM_INDEX}')
    elem_index = pd.read_parquet(PATH_ELEM_INDEX)
    elem_emb_existing = np.load(PATH_ELEM_EMB)
    done_keys = set(zip(elem_index['element'], elem_index['element_type']))
    embeddings_list = [elem_emb_existing]
    index_records = elem_index.to_dict('records')
    print(f'  resuming: {len(done_keys):,} elements already embedded')
else:
    done_keys = set()
    embeddings_list = []
    index_records = []
    print('No element-embedding checkpoint found; starting from scratch.')

remaining_elem = elements[~elements.apply(
    lambda r: (r['element'], r['element_type']) in done_keys, axis=1)
].reset_index(drop=True)
print(f'{len(remaining_elem):,} elements to embed')

n_total = len(remaining_elem)
since_flush = 0
new_batch_embs = []
new_batch_index = []

pbar = tqdm(range(0, n_total, BATCH_SIZE_ELEMS), desc='element embedding')
for batch_start in pbar:
    batch = remaining_elem.iloc[batch_start : batch_start + BATCH_SIZE_ELEMS]
    texts = batch['element'].tolist()
    embs = voyage_embed(texts)
    new_batch_embs.extend(embs)
    for _, row in batch.iterrows():
        new_batch_index.append({
            'element': row['element'],
            'element_type': row['element_type'],
            'is_narrative_element': bool(row['is_narrative_element']),
            'element_taxonomy': row['element_taxonomy'],
            'embedding_row_index': len(index_records) + len(new_batch_index),
        })
    since_flush += len(batch)
    time.sleep(REQUEST_SLEEP)

    if since_flush >= CHECKPOINT_EVERY:
        embeddings_list.append(np.array(new_batch_embs, dtype=np.float32))
        index_records.extend(new_batch_index)
        flushed = np.concatenate(embeddings_list, axis=0)
        np.save(PATH_ELEM_EMB, flushed)
        pd.DataFrame(index_records).to_parquet(PATH_ELEM_INDEX, index=False)
        embeddings_list = [flushed]
        new_batch_embs = []
        new_batch_index = []
        since_flush = 0
        pbar.set_postfix(flushed=len(flushed))

if new_batch_embs:
    embeddings_list.append(np.array(new_batch_embs, dtype=np.float32))
    index_records.extend(new_batch_index)
element_embeddings = np.concatenate(embeddings_list, axis=0)
np.save(PATH_ELEM_EMB, element_embeddings)
pd.DataFrame(index_records).to_parquet(PATH_ELEM_INDEX, index=False)

print(f'\nPhase B complete.')
print(f'  element embeddings: {element_embeddings.shape}  (elements × embedding_dim)')
print(f'  saved: {PATH_ELEM_EMB} and {PATH_ELEM_INDEX}')

No element-embedding checkpoint found; starting from scratch.
40,877 elements to embed


element embedding:   0%|          | 0/320 [00:00<?, ?it/s]


Phase B complete.
  element embeddings: (40877, 1024)  (elements × embedding_dim)
  saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/05_embeddings/element_embeddings.npy and /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/05_embeddings/element_embeddings_index.parquet


In [18]:
# ----------------------------------------------------------------------
# Phase C — BERTopic clustering on plot embeddings
#
# BERTopic discovers topics by combining UMAP dimensionality reduction
# with HDBSCAN density-based clustering and c-TF-IDF topic naming.
# We pass our pre-computed voyage embeddings rather than letting BERTopic
# embed via its default model.
#
# Output: per-plot topic assignment + a topic-summary table.
# ----------------------------------------------------------------------

# Re-load embeddings + index in case we skipped Phases A/B from a checkpoint
plot_embeddings = np.load(PATH_PLOT_EMB)
plot_index = pd.read_parquet(PATH_PLOT_INDEX)
print(f'Loaded plot embeddings: {plot_embeddings.shape}')

# Rebuild the per-plot plot text and genre for BERTopic naming + alignment
plot_index_with_meta = plot_index.merge(
    movies[['union_id','plot','genres_canonical']],
    on='union_id', how='left',
)
plot_index_with_meta = plot_index_with_meta.sort_values('embedding_row_index').reset_index(drop=True)
plot_texts = plot_index_with_meta['plot'].tolist()

print('\nRunning BERTopic ...')
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Pre-compute UMAP reduction so BERTopic uses our voyage embeddings.
# UMAP defaults: 5 components, neighbors=15, min_dist=0. The reduction
# is necessary because HDBSCAN does not scale to the raw embedding dim.
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=20260219,  # match the project's preregistered seed
)
hdbscan_model = HDBSCAN(
    min_cluster_size=BERTOPIC_MIN_CLUSTER_SIZE,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=False,  # too expensive for 46K docs
    verbose=False,
)

topics, _ = topic_model.fit_transform(plot_texts, embeddings=plot_embeddings)
plot_index_with_meta['bertopic_topic_id'] = topics

# Topic summary table
topic_info = topic_model.get_topic_info()
print(f'BERTopic produced {len(topic_info)} topic groups (including -1 = noise).')
print(f'  topic size distribution: '
      f'median={int(topic_info["Count"].median())}, '
      f'max={int(topic_info["Count"].max())}')

# Save per-plot topic assignments
bertopic_out = plot_index_with_meta[['union_id','embedding_row_index','bertopic_topic_id']].copy()
bertopic_out.to_parquet(PATH_BERTOPIC, index=False)
topic_info.to_parquet(OUT_DIR / 'bertopic_topic_info.parquet', index=False)
print(f'Saved: {PATH_BERTOPIC} and bertopic_topic_info.parquet')

Loaded plot embeddings: (45958, 1024)

Running BERTopic ...
BERTopic produced 51 topic groups (including -1 = noise).
  topic size distribution: median=262, max=19954
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/05_embeddings/bertopic_topics.parquet and bertopic_topic_info.parquet


In [ ]:
# ----------------------------------------------------------------------
# Phase D — Gaussian Mixture Model clustering with adaptive-stop BIC
#
# Sweep n_components from GMM_N_MIN upward. Track the running minimum
# BIC. Stop early if BIC has not improved for GMM_PATIENCE consecutive
# steps (a true minimum has been passed). If we reach GMM_N_MAX without
# stopping, the BIC is still descending — report a warning so the
# downstream interpretation does not treat the upper-bound value as a
# discovered optimum.
#
# PCA to 50 components first for tractability. voyage-4-large outputs
# 1024-dim embeddings; full-dim GMM is slow and the top-50 PCs preserve
# the bulk of the variance.
# ----------------------------------------------------------------------
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

print(f'Reducing plot embeddings to 50 PCA components for GMM efficiency...')
pca = PCA(n_components=50, random_state=20260219)
plot_emb_pca = pca.fit_transform(plot_embeddings)
print(f'  PCA explained variance (top 5): {pca.explained_variance_ratio_[:5].round(3).tolist()}')
print(f'  cumulative explained variance : {pca.explained_variance_ratio_.sum():.3f}')

print(f'\nGMM BIC adaptive-stop sweep: n={GMM_N_MIN} ... {GMM_N_MAX}, patience={GMM_PATIENCE}')
bic_scores = []
best_bic = np.inf
best_n = None
since_improvement = 0
hit_upper_bound = True

for n in tqdm(range(GMM_N_MIN, GMM_N_MAX + 1), desc='GMM BIC sweep'):
    gmm = GaussianMixture(
        n_components=n,
        covariance_type='diag',
        random_state=20260219,
        max_iter=200,
        n_init=1,
    )
    gmm.fit(plot_emb_pca)
    bic = gmm.bic(plot_emb_pca)
    bic_scores.append({'n_components': n, 'bic': bic})

    if bic < best_bic:
        best_bic = bic
        best_n = n
        since_improvement = 0
    else:
        since_improvement += 1

    if since_improvement >= GMM_PATIENCE:
        print(f'\n  BIC has not improved for {GMM_PATIENCE} consecutive steps. Stopping at n={n}.')
        print(f'  Selected n_components = {best_n} (BIC = {best_bic:,.0f})')
        hit_upper_bound = False
        break

if hit_upper_bound:
    print(f'\n  WARNING: BIC kept decreasing through GMM_N_MAX = {GMM_N_MAX}.')
    print(f'  The selected n_components = {best_n} is the upper bound and may not be a true optimum.')
    print(f'  Consider extending GMM_N_MAX further.')

bic_df = pd.DataFrame(bic_scores)
bic_df['delta_from_prev'] = bic_df['bic'].diff()
bic_df['delta_from_running_min'] = bic_df['bic'] - bic_df['bic'].cummin()
print(f'\nBIC sweep summary (final 10 rows):')
print(bic_df.tail(10).to_string(index=False, float_format='%.0f'))

print(f'\nRefitting GMM at n_components = {best_n} (with 3 inits for stability)...')
gmm = GaussianMixture(
    n_components=best_n,
    covariance_type='diag',
    random_state=20260219,
    max_iter=200,
    n_init=3,
)
gmm.fit(plot_emb_pca)
gmm_assignments = gmm.predict(plot_emb_pca)
gmm_proba = gmm.predict_proba(plot_emb_pca)

gmm_out = plot_index_with_meta[['union_id','embedding_row_index']].copy()
gmm_out['gmm_cluster'] = gmm_assignments
gmm_out['gmm_top_proba'] = gmm_proba.max(axis=1)
gmm_out['gmm_n_components'] = best_n
gmm_out['gmm_bic_hit_upper_bound'] = hit_upper_bound
gmm_out.to_parquet(PATH_GMM, index=False)
bic_df.to_parquet(OUT_DIR / 'gmm_bic_sweep.parquet', index=False)
print(f'Saved: {PATH_GMM} and gmm_bic_sweep.parquet')

Reducing plot embeddings to 50 PCA components for GMM efficiency...
  PCA explained variance (top 5): [0.032999999821186066, 0.028999999165534973, 0.02199999988079071, 0.019999999552965164, 0.01899999938905239]
  cumulative explained variance : 0.435

GMM BIC adaptive-stop sweep: n=8 ... 80, patience=8


GMM BIC sweep:   0%|          | 0/73 [00:00<?, ?it/s]


  The selected n_components = 80 is the upper bound and may not be a true optimum.
  Consider extending GMM_N_MAX further.

BIC sweep summary (final 10 rows):
 n_components      bic  delta_from_prev  delta_from_running_min
           71 -5489280            -1706                       0
           72 -5489950             -670                       0
           73 -5489817              133                     133
           74 -5488375             1441                    1574
           75 -5491307            -2931                       0
           76 -5490019             1288                    1288
           77 -5491622            -1603                       0
           78 -5491900             -278                       0
           79 -5492852             -953                       0
           80 -5495197            -2345                       0

Refitting GMM at n_components = 80 (with 3 inits for stability)...
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene 

In [ ]:
# ----------------------------------------------------------------------
# Phase E — Cluster-genre alignment
#
# For both BERTopic and GMM clusterings, compute alignment with the
# seventeen canonical genres:
#   - Adjusted Rand Index (ARI): chance-corrected similarity of two partitions
#   - Normalized Mutual Information (NMI): information-theoretic agreement
#   - Confusion matrix: per-genre share of each discovered cluster
#
# Since genres are multi-label and clusters are single-label, we expand
# each film into one row per (film, genre) and use the genre label as
# the "true" assignment. This inflates the row count but lets standard
# clustering metrics apply.
# ----------------------------------------------------------------------
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Build the multi-label expansion: one row per (film, genre)
exploded = (plot_index_with_meta[['union_id','embedding_row_index','genres_canonical','bertopic_topic_id']]
            .merge(gmm_out[['union_id','gmm_cluster']], on='union_id', how='left')
            .explode('genres_canonical')
            .rename(columns={'genres_canonical':'genre'})
            .dropna(subset=['genre']))
exploded = exploded[exploded['genre'].isin(CANONICAL_GENRES)]

print(f'Expanded (film, genre) rows: {len(exploded):,}')

# Drop -1 BERTopic noise label for ARI/NMI (BERTopic's noise convention)
bt = exploded[exploded['bertopic_topic_id'] != -1]
ari_bertopic = adjusted_rand_score(bt['genre'], bt['bertopic_topic_id'])
nmi_bertopic = normalized_mutual_info_score(bt['genre'], bt['bertopic_topic_id'])

ari_gmm = adjusted_rand_score(exploded['genre'], exploded['gmm_cluster'])
nmi_gmm = normalized_mutual_info_score(exploded['genre'], exploded['gmm_cluster'])

print(f'\nBERTopic vs canonical genres (excluding -1 noise):')
print(f'  Adjusted Rand Index           : {ari_bertopic:.4f}')
print(f'  Normalized Mutual Information : {nmi_bertopic:.4f}')
print(f'  noise share (% films in -1)   : {(exploded["bertopic_topic_id"]==-1).mean()*100:.1f}%')

print(f'\nGMM (BIC-selected n={best_n}) vs canonical genres:')
print(f'  Adjusted Rand Index           : {ari_gmm:.4f}')
print(f'  Normalized Mutual Information : {nmi_gmm:.4f}')

# Confusion table: which clusters concentrate which genres?
print('\nGMM cluster × genre share table (top 3 genres per cluster):')
cm = pd.crosstab(exploded['gmm_cluster'], exploded['genre'])
cm_norm = cm.div(cm.sum(axis=1), axis=0)
for c in cm.index:
    top3 = cm_norm.loc[c].nlargest(3)
    print(f'  cluster {c:>2}: ' + ', '.join(f'{g}({s*100:.0f}%)' for g, s in top3.items()))

alignment_records = [
    {'clustering': 'bertopic', 'n_clusters': bt['bertopic_topic_id'].nunique(),
     'noise_share': (exploded['bertopic_topic_id']==-1).mean(),
     'adjusted_rand': ari_bertopic, 'nmi': nmi_bertopic},
    {'clustering': 'gmm', 'n_clusters': best_n, 'noise_share': 0.0,
     'adjusted_rand': ari_gmm, 'nmi': nmi_gmm},
]
pd.DataFrame(alignment_records).to_parquet(PATH_ALIGNMENT, index=False)
print(f'\nSaved: {PATH_ALIGNMENT}')

Expanded (film, genre) rows: 100,130

BERTopic vs canonical genres (excluding -1 noise):
  Adjusted Rand Index           : 0.0611
  Normalized Mutual Information : 0.1848
  noise share (% films in -1)   : 44.7%

GMM (BIC-selected n=80) vs canonical genres:
  Adjusted Rand Index           : 0.0196
  Normalized Mutual Information : 0.1489

GMM cluster × genre share table (top 3 genres per cluster):
  cluster  0: Drama(31%), Comedy(22%), Romance(14%)
  cluster  1: Crime(22%), Action(18%), Thriller(18%)
  cluster  2: Animation(40%), Comedy(28%), Family(25%)
  cluster  3: Comedy(32%), Drama(24%), Romance(8%)
  cluster  4: Drama(37%), Comedy(14%), Romance(11%)
  cluster  5: Adventure(29%), Action(14%), Drama(12%)
  cluster  6: Thriller(23%), Action(20%), Drama(16%)
  cluster  7: Action(28%), Drama(24%), Romance(11%)
  cluster  8: Drama(42%), Romance(22%), Comedy(19%)
  cluster  9: Drama(27%), Comedy(14%), Crime(11%)
  cluster 10: Horror(37%), Thriller(19%), Comedy(11%)
  cluster 11: Drama(29

In [19]:
# ----------------------------------------------------------------------
# Phase F — Per-genre semantic validation
#
# Three validation arms, all computed against the per-genre embedding
# centroids:
#
#   1. Top-50 stock narrative elements vs random-50 narrative baseline.
#      Tests whether the framework\'s count-based ranking aligns with the
#      genre\'s semantic core. Reports mean cosine, gap, Cohen\'s d, and
#      Welch t-test p-value.
#
#   2. Top-50 narrative vs top-50 FILTERED (non-narrative).
#      Tests whether 03b\'s narrative-vs-noise classification is doing
#      real semantic work. The narrative-minus-filtered gap should be
#      positive across the board; a negative result for a specific genre
#      is doctrinally interesting (it would say that genre\'s identity
#      is carried partly by what 03b classifies as filtered).
#
#   3. Per-(element, genre) cosine table preserved as-is for downstream
#      paper figures.
# ----------------------------------------------------------------------
from scipy.stats import ttest_ind

plot_embeddings = np.load(PATH_PLOT_EMB)
elem_embeddings = np.load(PATH_ELEM_EMB)
plot_index = pd.read_parquet(PATH_PLOT_INDEX)
elem_index = pd.read_parquet(PATH_ELEM_INDEX)
plot_index = plot_index.merge(movies[['union_id','genres_canonical']], on='union_id', how='left')
plot_index['genres_canonical'] = plot_index['genres_canonical'].apply(
    lambda gs: gs if isinstance(gs, list) else (list(gs) if hasattr(gs, '__len__') else [])
)

def l2_normalize(M):
    n = np.linalg.norm(M, axis=1, keepdims=True)
    n = np.where(n == 0, 1.0, n)
    return M / n

plot_emb_n = l2_normalize(plot_embeddings)
elem_emb_n = l2_normalize(elem_embeddings)

# Per-genre centroids
print('Computing per-genre semantic centroids...')
genre_centroids = {}
for g in CANONICAL_GENRES:
    mask = plot_index['genres_canonical'].apply(lambda gs: g in gs).values
    if mask.sum() < 10:
        continue
    centroid = plot_emb_n[mask].mean(axis=0)
    centroid = centroid / max(np.linalg.norm(centroid), 1e-12)
    genre_centroids[g] = centroid
    print(f'  {g:<18} {int(mask.sum()):>6,} films')

# Per-(element, genre) similarity matrix
genres_in_order = [g for g in CANONICAL_GENRES if g in genre_centroids]
centroid_matrix = np.array([genre_centroids[g] for g in genres_in_order], dtype=np.float32)
sim_matrix = elem_emb_n @ centroid_matrix.T  # (n_elements, n_genres)

print('\nBuilding per-(element, genre) cosine table...')
sim_long_rows = []
for ei in range(len(elem_index)):
    row = elem_index.iloc[ei]
    row_idx = row['embedding_row_index']
    for gi, g in enumerate(genres_in_order):
        sim_long_rows.append({
            'element': row['element'],
            'element_type': row['element_type'],
            'is_narrative_element': row['is_narrative_element'],
            'element_taxonomy': row['element_taxonomy'],
            'genre': g,
            'cosine_to_centroid': float(sim_matrix[row_idx, gi]),
        })
elem_genre_sim = pd.DataFrame(sim_long_rows)
elem_genre_sim.to_parquet(PATH_CENTROIDS, index=False)
print(f'Saved per-(element, genre) similarities: {PATH_CENTROIDS}  ({len(elem_genre_sim):,} rows)')

# Load classified stockiness for the top-50 selection
stk_cls = pd.read_parquet(PATH_ENRICHED)

# Arm 1: top-50 narrative vs random-50 baseline
print('\nArm 1: top-50 narrative vs random-50 narrative baseline (the headline)')
rng = np.random.default_rng(20260219)
arm1_rows = []
for g in genres_in_order:
    sim_g = elem_genre_sim[elem_genre_sim['genre']==g]
    top50 = (stk_cls[(stk_cls['genre']==g) & (stk_cls['is_narrative_element'])]
             .nlargest(50, 'ensemble')[['element','element_type']].drop_duplicates())
    top_sim = sim_g.merge(top50, on=['element','element_type'])['cosine_to_centroid'].values

    # Random-50 narrative baseline, excluding the top-50 themselves
    narr_pool = elem_index[elem_index['is_narrative_element']]
    top_keys = set(zip(top50['element'], top50['element_type']))
    pool = narr_pool[~narr_pool.apply(lambda r: (r['element'], r['element_type']) in top_keys, axis=1)]
    rand_idx = rng.choice(len(pool), size=min(50, len(pool)), replace=False)
    rand50 = pool.iloc[rand_idx][['element','element_type']]
    rand_sim = sim_g.merge(rand50, on=['element','element_type'])['cosine_to_centroid'].values

    if len(top_sim) == 0 or len(rand_sim) == 0:
        continue
    gap = top_sim.mean() - rand_sim.mean()
    pooled_std = np.sqrt((top_sim.var(ddof=1) + rand_sim.var(ddof=1)) / 2)
    cohens_d = gap / pooled_std if pooled_std > 0 else 0.0
    t_stat, p_val = ttest_ind(top_sim, rand_sim, equal_var=False)
    arm1_rows.append({
        'genre': g,
        'n_top': len(top_sim), 'n_rand': len(rand_sim),
        'top50_mean_cosine': top_sim.mean(),
        'random50_mean_cosine': rand_sim.mean(),
        'gap': gap,
        'cohens_d': cohens_d,
        't_stat': t_stat,
        'p_value': p_val,
    })

validation_df = pd.DataFrame(arm1_rows)
validation_df.to_parquet(OUT_DIR / 'semantic_validation.parquet', index=False)
print(validation_df.sort_values('gap', ascending=False).to_string(
    index=False, float_format='%.4f'))

# Arm 2: top-50 narrative vs top-50 filtered
print('\nArm 2: top-50 narrative vs top-50 FILTERED (03b classification validation)')
print('  Tests whether 03b\'s narrative-vs-noise distinction is semantically meaningful.')
print('  A positive narrative-minus-filtered gap means narrative elements sit')
print('  closer to the genre centroid than filtered elements (the desired result).')
arm2_rows = []
for g in genres_in_order:
    sim_g = elem_genre_sim[elem_genre_sim['genre']==g]
    top50_narr = (stk_cls[(stk_cls['genre']==g) & (stk_cls['is_narrative_element'])]
                  .nlargest(50, 'ensemble')[['element','element_type']].drop_duplicates())
    top50_filt = (stk_cls[(stk_cls['genre']==g) & (~stk_cls['is_narrative_element'])]
                  .nlargest(50, 'ensemble')[['element','element_type']].drop_duplicates())
    narr_sim = sim_g.merge(top50_narr, on=['element','element_type'])['cosine_to_centroid'].values
    filt_sim = sim_g.merge(top50_filt, on=['element','element_type'])['cosine_to_centroid'].values
    if len(narr_sim) == 0 or len(filt_sim) == 0:
        continue
    diff = narr_sim.mean() - filt_sim.mean()
    pooled_std = np.sqrt((narr_sim.var(ddof=1) + filt_sim.var(ddof=1)) / 2)
    cohens_d = diff / pooled_std if pooled_std > 0 else 0.0
    t_stat, p_val = ttest_ind(narr_sim, filt_sim, equal_var=False)
    arm2_rows.append({
        'genre': g,
        'n_narr': len(narr_sim), 'n_filt': len(filt_sim),
        'top_narr_mean': narr_sim.mean(),
        'top_filt_mean': filt_sim.mean(),
        'narr_minus_filt': diff,
        'cohens_d': cohens_d,
        't_stat': t_stat,
        'p_value': p_val,
    })
filter_df = pd.DataFrame(arm2_rows).sort_values('narr_minus_filt', ascending=False)
filter_df.to_parquet(OUT_DIR / 'filter_validation.parquet', index=False)
print(filter_df.to_string(index=False, float_format='%.4f'))

# Flag any genre where filtered actually exceeds narrative
filter_anomalies = filter_df[filter_df['narr_minus_filt'] < 0]
if len(filter_anomalies):
    print(f'\n  ANOMALY: {len(filter_anomalies)} genre(s) where filtered elements are')
    print(f'  semantically CLOSER to the centroid than narrative elements:')
    for _, r in filter_anomalies.iterrows():
        print(f'    {r["genre"]:<18} narr={r["top_narr_mean"]:.4f}, filt={r["top_filt_mean"]:.4f}, '
              f'diff={r["narr_minus_filt"]:+.4f}')
    print(f'  Doctrinal reading: in these genres, the genre\'s semantic identity is partly')
    print(f'  carried by what 03b classifies as character_name / proper_noun_other (franchise')
    print(f'  characters, named places, etc.) rather than by generic narrative elements.')
else:
    print(f'\n  All genres: narrative > filtered. 03b\'s classification is semantically validated.')

Computing per-genre semantic centroids...
  Action              7,775 films
  Adventure           6,146 films
  Animation           2,835 films
  Comedy             14,724 films
  Crime               5,564 films
  Drama              22,364 films
  Family              4,164 films
  Fantasy             2,429 films
  History             2,882 films
  Horror              4,363 films
  Music               2,926 films
  Mystery             2,681 films
  Romance             8,220 films
  Science Fiction     2,590 films
  Thriller            7,447 films
  War                 1,658 films
  Western             1,362 films

Building per-(element, genre) cosine table...
Saved per-(element, genre) similarities: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/05_embeddings/genre_centroids.parquet  (694,909 rows)

Arm 1: top-50 narrative vs random-50 narrative baseline (the headline)
          genre  n_top  n_rand  top50_mean_cosine  random50_mea

In [20]:
# ----------------------------------------------------------------------
# Write the human-readable report.
#
# Layout:
#   - Header (model, sizes)
#   - GMM BIC sweep (with adaptive-stop status)
#   - Cluster-genre alignment (BERTopic vs GMM)
#   - Arm 1 validation: top-50 narrative vs random-50, with Cohen\'s d and p
#   - Arm 2 validation: narrative vs filtered (03b classification check)
#   - Anomaly callout (any genre where filtered > narrative)
# ----------------------------------------------------------------------
lines = []
def out(line=''):
    print(line); lines.append(line)

out('=' * 70)
out('NOTEBOOK 05: EMBEDDINGS AND TOPIC MODELING — SEMANTIC VALIDATION')
out('=' * 70)
out(f'embedding model : {MODEL_NAME}')
out(f'plot embeddings : {plot_embeddings.shape[0]:,} × {plot_embeddings.shape[1]}')
out(f'element embeds  : {elem_embeddings.shape[0]:,} × {elem_embeddings.shape[1]}')
out('')

out('=' * 70)
out('GMM BIC SWEEP (adaptive-stop)')
out('=' * 70)
if gmm_out["gmm_bic_hit_upper_bound"].iloc[0]:
    out(f'  WARNING: BIC kept decreasing through GMM_N_MAX = {GMM_N_MAX}')
    out(f'  Selected n_components = {best_n} (the upper bound).')
    out(f'  The true BIC minimum lies above {GMM_N_MAX}. Consider extending GMM_N_MAX.')
else:
    out(f'  Adaptive stop fired: BIC did not improve for {GMM_PATIENCE} consecutive steps.')
    out(f'  Selected n_components = {best_n} (BIC = {best_bic:,.0f}).')
out(f'\n  BIC sweep (n_components, BIC, delta-from-running-min):')
for _, r in bic_df.iterrows():
    marker = '  ← min' if int(r["n_components"]) == best_n else ''
    out(f'    n={int(r["n_components"]):>3}  BIC={r["bic"]:>15,.0f}  '
        f'Δ_from_min={r["delta_from_running_min"]:>12,.0f}{marker}')
out('')

out('=' * 70)
out('CLUSTER-GENRE ALIGNMENT')
out('=' * 70)
out('Both clusterings are compared against the seventeen canonical genres.')
out('Adjusted Rand Index = 0 means random alignment; 1 means perfect.')
out('NMI similarly bounded [0, 1].')
out('')
out(f'BERTopic   : ARI = {ari_bertopic:.4f}, NMI = {nmi_bertopic:.4f}, '
    f'noise = {(exploded["bertopic_topic_id"]==-1).mean()*100:.1f}%, '
    f'n_clusters = {bt["bertopic_topic_id"].nunique()}')
out(f'GMM (n={best_n}): ARI = {ari_gmm:.4f}, NMI = {nmi_gmm:.4f}')
out('')

out('=' * 70)
out('ARM 1 — TOP-50 NARRATIVE vs RANDOM-50: cosine to genre centroid')
out('=' * 70)
out('A positive gap means top-stock narrative elements sit closer to the')
out('genre centroid than random narrative elements. Larger gaps and Cohen\'s d')
out('values indicate stronger validation of the framework\'s count-based')
out('ranking by an independent semantic representation.')
out('')
out(f'  {"genre":<18} {"top50":>8} {"rand50":>8} {"gap":>9} '
    f'{"cohens_d":>10} {"p_value":>10}')
for _, r in validation_df.sort_values('gap', ascending=False).iterrows():
    star = ' ***' if r['p_value'] < 0.001 else (' **' if r['p_value'] < 0.01 else (' *' if r['p_value'] < 0.05 else ''))
    out(f'  {r["genre"]:<18} {r["top50_mean_cosine"]:>8.4f} '
        f'{r["random50_mean_cosine"]:>8.4f} {r["gap"]:>+9.4f} '
        f'{r["cohens_d"]:>+10.3f} {r["p_value"]:>10.3e}{star}')
out('')
out(f'  Mean gap across genres : {validation_df["gap"].mean():+.4f}')
out(f'  Median gap             : {validation_df["gap"].median():+.4f}')
out(f'  Mean Cohen\'s d         : {validation_df["cohens_d"].mean():+.3f}')
out(f'  Genres with p < 0.05   : {(validation_df["p_value"] < 0.05).sum()} / {len(validation_df)}')
out(f'  Genres with p < 0.001  : {(validation_df["p_value"] < 0.001).sum()} / {len(validation_df)}')
out(f'  Genres with d ≥ 0.8 (large) : {(validation_df["cohens_d"] >= 0.8).sum()} / {len(validation_df)}')
out(f'  Genres with d ≥ 0.5 (medium): {(validation_df["cohens_d"] >= 0.5).sum()} / {len(validation_df)}')
out('')

out('=' * 70)
out('ARM 2 — TOP-50 NARRATIVE vs TOP-50 FILTERED: 03b classification validity')
out('=' * 70)
out('A positive narrative-minus-filtered gap means narrative elements sit')
out('closer to the genre centroid than 03b-classified filtered elements.')
out('A consistent positive result confirms that 03b\'s narrative-vs-noise')
out('classification is doing real semantic work, not just count-statistical work.')
out('')
out(f'  {"genre":<18} {"narr":>8} {"filt":>8} {"diff":>9} '
    f'{"cohens_d":>10} {"p_value":>10}')
for _, r in filter_df.iterrows():
    star = ' ***' if r['p_value'] < 0.001 else (' **' if r['p_value'] < 0.01 else (' *' if r['p_value'] < 0.05 else ''))
    out(f'  {r["genre"]:<18} {r["top_narr_mean"]:>8.4f} '
        f'{r["top_filt_mean"]:>8.4f} {r["narr_minus_filt"]:>+9.4f} '
        f'{r["cohens_d"]:>+10.3f} {r["p_value"]:>10.3e}{star}')
out('')
out(f'  Mean narrative-minus-filtered: {filter_df["narr_minus_filt"].mean():+.4f}')
out(f'  Genres where narrative > filtered: {(filter_df["narr_minus_filt"] > 0).sum()} / {len(filter_df)}')

# Anomalies
filter_anomalies_in_report = filter_df[filter_df['narr_minus_filt'] < 0]
if len(filter_anomalies_in_report):
    out('')
    out('  ANOMALY: genre(s) where filtered elements are CLOSER to the centroid')
    out('  than narrative elements. Doctrinal reading: that genre\'s semantic')
    out('  identity is partly carried by named entities (franchise characters,')
    out('  proper-noun work titles) that 03b classifies as non-narrative.')
    for _, r in filter_anomalies_in_report.iterrows():
        out(f'    {r["genre"]:<18} narr={r["top_narr_mean"]:.4f}, '
            f'filt={r["top_filt_mean"]:.4f}, diff={r["narr_minus_filt"]:+.4f}')

PATH_REPORT.write_text('\n'.join(lines))
print(f'\nReport written to {PATH_REPORT}')

NOTEBOOK 05: EMBEDDINGS AND TOPIC MODELING — SEMANTIC VALIDATION
embedding model : voyage-4-large
plot embeddings : 45,958 × 1024
element embeds  : 40,877 × 1024

GMM BIC SWEEP (adaptive-stop)
  Selected n_components = 80 (the upper bound).
  The true BIC minimum lies above 80. Consider extending GMM_N_MAX.

  BIC sweep (n_components, BIC, delta-from-running-min):
    n=  8  BIC=     -5,298,881  Δ_from_min=           0
    n=  9  BIC=     -5,306,057  Δ_from_min=           0
    n= 10  BIC=     -5,320,531  Δ_from_min=           0
    n= 11  BIC=     -5,332,322  Δ_from_min=           0
    n= 12  BIC=     -5,344,260  Δ_from_min=           0
    n= 13  BIC=     -5,349,454  Δ_from_min=           0
    n= 14  BIC=     -5,361,249  Δ_from_min=           0
    n= 15  BIC=     -5,369,012  Δ_from_min=           0
    n= 16  BIC=     -5,368,868  Δ_from_min=         144
    n= 17  BIC=     -5,377,652  Δ_from_min=           0
    n= 18  BIC=     -5,380,496  Δ_from_min=           0
    n= 19  BIC=  

## Outputs produced

- **`plot_embeddings.npy`** + **`plot_embeddings_index.parquet`** — the (45,958, embedding_dim) plot embedding matrix and a parquet mapping each `union_id` to its row index.
- **`element_embeddings.npy`** + **`element_embeddings_index.parquet`** — the (41,186, embedding_dim) element embedding matrix and a parquet mapping each `(element, element_type)` pair to its row index, plus the 03b classification.
- **`bertopic_topics.parquet`** — per-plot BERTopic topic assignment. -1 = noise.
- **`bertopic_topic_info.parquet`** — per-topic size and c-TF-IDF representation.
- **`gmm_clusters.parquet`** — per-plot GMM hard assignment plus top-component probability, for the BIC-selected number of components.
- **`gmm_bic_sweep.parquet`** — BIC values across the sweep range, for the supplementary figure.
- **`genre_centroids.parquet`** — per-(element, genre) cosine similarity to each genre's semantic centroid. The headline table for the per-genre validation check.
- **`semantic_validation.parquet`** — per-genre summary: mean cosine of top-50 stock elements vs random-50 baseline, plus the gap.
- **`cluster_genre_alignment.parquet`** — ARI and NMI scores for both clusterings.
- **`semantic_validation_report.txt`** — human-readable report with cluster-genre alignment, GMM BIC sweep, and the per-genre validation table.

## How to read the validation

The framework's headline doctrinal claim is that its element-level stockiness scores capture genre-conventional vocabulary. Notebook 05 is a sanity check on that claim from a completely different angle. The two layers of evidence:

1. **Cluster-genre alignment.** ARI and NMI between BERTopic (or GMM) clusters and the seventeen canonical genres. Higher values mean the embeddings independently recover the genre structure the framework relies on. Modest values are expected (genres are multi-label, embeddings are single-label assignments). Strong values would be unusually high for any real corpus.

2. **Per-genre top-stock validation.** For each genre, the mean cosine similarity of the framework's top-50 stock elements to the genre centroid, compared against a random-50 baseline. A consistently positive gap across all seventeen genres is what validates the framework. A negative gap on any genre would be doctrinally concerning and worth investigating.

The first layer tests whether the genre labels are real. The second layer tests whether the framework's per-element scoring agrees with what semantic embeddings see as central to each genre.

## What can go wrong

- **The voyage API fails or rate-limits.** The notebook caches partial embeddings every 1000 items so a crashed run resumes cleanly.
- **The GMM BIC sweep is slow.** Reducing to 50 PCA components keeps each fit fast. If the sweep is still too slow on large embeddings, reduce GMM_N_RANGE.
- **BERTopic creates too many noise points.** Lower min_cluster_size to allow smaller clusters, or accept noise share as a finding worth reporting.
- **An element appears identical to another after lowercasing.** Each (element, element_type) pair is embedded separately, but identical strings will get identical embeddings; the downstream cosine table preserves both rows.

## Sensitivity checks to consider for the paper

1. Re-run BERTopic with a different `min_cluster_size` (50, 200) to confirm the discovered topics are robust to the choice.
2. Restrict the centroid computation to films tagged with only one canonical genre (single-label films), to remove cross-genre contamination from the centroid. Likely produces sharper centroids.
3. Compute the semantic validation gap separately for the four 03b filter categories (character_name, place_name, proper_noun_other, generic_or_empty). The filtered elements should NOT show a positive gap; that would validate the 03b filtering.